In [3]:
import os
import glob
import subprocess
import json
import csv
import re

def get_video_info(video_path):
    """Get video information using ffprobe"""
    try:
        # Get duration using ffprobe
        cmd = [
            'ffprobe', 
            '-v', 'error', 
            '-select_streams', 'v:0', 
            '-show_entries', 'stream=duration', 
            '-of', 'json', 
            video_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        info = json.loads(result.stdout)
        
        # Extract duration
        duration = float(info['streams'][0]['duration'])
        return duration
    
    except Exception as e:
        print(f"Error getting video info: {e}")
        return 0

def parse_csv_annotation(csv_file):
    """Parse the CSV annotation file and return list of gestures"""
    gestures = []
    
    try:
        with open(csv_file, 'r', encoding='utf-8') as f:
            # Detect column names for time and gesture
            reader = csv.reader(f)
            header = next(reader)
            
            # Find column indices
            start_col = -1
            end_col = -1
            gesture_col = -1
            
            for i, col_name in enumerate(header):
                if 'begin time' in col_name.lower() or 'start' in col_name.lower():
                    start_col = i
                elif 'end time' in col_name.lower() or 'stop' in col_name.lower():
                    end_col = i
                elif 'gesture' in col_name.lower() or 'phrase' in col_name.lower():
                    gesture_col = i
            
            if start_col == -1 or end_col == -1 or gesture_col == -1:
                print(f"Could not identify required columns in {csv_file}")
                print(f"Header: {header}")
                return []
            
            # Reset file pointer and skip header
            f.seek(0)
            next(reader)
            
            # Parse rows
            for row in reader:
                try:
                    if len(row) > max(start_col, end_col, gesture_col):
                        start_time = int(row[start_col]) / 1000.0  # Convert ms to seconds
                        end_time = int(row[end_col]) / 1000.0      # Convert ms to seconds
                        gesture_type = row[gesture_col].strip()
                        
                        if gesture_type and end_time > start_time:
                            gestures.append({
                                'start_time': start_time,
                                'end_time': end_time,
                                'type': gesture_type
                            })
                except ValueError as e:
                    print(f"Error parsing row: {row}, Error: {e}")
                    continue
                
    except Exception as e:
        print(f"Error processing CSV file {csv_file}: {e}")
    
    return gestures

def extract_clip_with_padding(input_file, output_file, start_time, end_time, total_duration, padding=1.0):
    """Extract clip using ffmpeg with padding before and after"""
    try:
        # Add padding and ensure we don't go out of bounds
        padded_start = max(0, start_time - padding)
        padded_end = min(total_duration, end_time + padding)
        clip_duration = padded_end - padded_start
        
        cmd = [
            'ffmpeg', '-y',
            '-ss', str(padded_start),
            '-i', input_file,
            '-t', str(clip_duration),
            '-c:v', 'libx264',
            '-c:a', 'aac',
            '-strict', 'experimental',
            output_file
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if os.path.exists(output_file) and os.path.getsize(output_file) > 1024:
            return True
        else:
            print(f"Failed to create valid output file: {output_file}")
            print(f"FFmpeg output: {result.stderr}")
            if os.path.exists(output_file):
                os.remove(output_file)
            return False
            
    except Exception as e:
        print(f"Error extracting clip: {e}")
        if os.path.exists(output_file):
            os.remove(output_file)
        return False

def clean_gesture_name(gesture_name):
    """Clean gesture name for use in filenames"""
    # Replace non-alphanumeric characters with underscore
    clean_name = re.sub(r'[^a-zA-Z0-9]', '_', gesture_name)
    # Remove consecutive underscores
    clean_name = re.sub(r'_+', '_', clean_name)
    # Remove leading and trailing underscores
    clean_name = clean_name.strip('_')
    # Ensure name is not empty
    if not clean_name:
        clean_name = "unknown"
    return clean_name

def process_video(video_path, annotations_folder, output_folder):
    """Process a single video file with its corresponding annotation file"""
    # Create output folder for gestures
    gesture_folder = os.path.join(output_folder, 'gestures')
    os.makedirs(gesture_folder, exist_ok=True)
    
    # Extract video ID from filename (e.g., "V7" from "V7.mp4")
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    
    # Get the annotation file path
    annotation_file = os.path.join(annotations_folder, f"{video_id}.csv")
    
    if not os.path.exists(annotation_file):
        print(f"No annotation file found for {video_path}")
        return
    
    try:
        # Get video duration
        total_duration = get_video_info(video_path)
        
        if total_duration == 0:
            print(f"Could not determine duration for {video_path}")
            return
            
        print(f"Video duration: {total_duration:.2f} seconds")
        
        # Parse annotation file
        gestures = parse_csv_annotation(annotation_file)
        print(f"Found {len(gestures)} gestures in {video_id}")
        
        # Filter valid gestures (within video duration)
        valid_gestures = [g for g in gestures if g['end_time'] <= total_duration and g['start_time'] >= 0]
        
        # Process each gesture
        for idx, gesture in enumerate(valid_gestures):
            # Clean gesture type for filename
            safe_type = clean_gesture_name(gesture['type'])
            
            # Output filename using DATASET_VIDEOID_INDEX_TYPE format
            gesture_output = os.path.join(
                gesture_folder, 
                f'SAGA_{video_id}_{idx:04d}_{safe_type}.mp4'
            )
            
            if not os.path.exists(gesture_output):
                print(f"Extracting gesture clip {idx} from {video_id}: "
                      f"{gesture['start_time']:.2f}s - {gesture['end_time']:.2f}s "
                      f"(Type: {gesture['type']}) with 1s padding")
                      
                if extract_clip_with_padding(video_path, gesture_output, 
                                          gesture['start_time'], gesture['end_time'], 
                                          total_duration, padding=1.0):
                    print(f"Successfully extracted gesture clip: {gesture_output}")
                else:
                    print(f"Failed to extract gesture clip: {gesture_output}")
                
    except Exception as e:
        print(f"Error processing video {video_path}: {e}")

def main():
    # Configuration paths
    videos_folder = './OriginalVideos/'
    annotations_folder = './OriginalCodings/'
    output_folder = './analysis/'
    
    # Get list of videos
    video_extensions = ['.mp4', '.avi', '.mov']
    video_list = []
    
    for ext in video_extensions:
        video_list.extend(glob.glob(os.path.join(videos_folder, f'*{ext}')))
    
    print(f"Found {len(video_list)} videos to process")
    
    # Process each video
    for video_path in sorted(video_list):
        print(f"\nProcessing video: {video_path}")
        process_video(video_path, annotations_folder, output_folder)

if __name__ == "__main__":
    main()

Found 6 videos to process

Processing video: ./OriginalVideos\V07.mp4
Video duration: 520.04 seconds
Found 152 gestures in V07
Extracting gesture clip 0 from V07: 7.94s - 9.21s (Type: move) with 1s padding
Successfully extracted gesture clip: ./analysis/gestures\SAGA_V07_0000_move.mp4
Extracting gesture clip 1 from V07: 9.57s - 11.26s (Type: move) with 1s padding
Successfully extracted gesture clip: ./analysis/gestures\SAGA_V07_0001_move.mp4
Extracting gesture clip 2 from V07: 14.66s - 15.85s (Type: iconic) with 1s padding
Successfully extracted gesture clip: ./analysis/gestures\SAGA_V07_0002_iconic.mp4
Extracting gesture clip 3 from V07: 18.21s - 19.82s (Type: iconic) with 1s padding
Successfully extracted gesture clip: ./analysis/gestures\SAGA_V07_0003_iconic.mp4
Extracting gesture clip 4 from V07: 19.82s - 22.44s (Type: iconic) with 1s padding
Successfully extracted gesture clip: ./analysis/gestures\SAGA_V07_0004_iconic.mp4
Extracting gesture clip 5 from V07: 22.44s - 24.15s (Type: 